In [3]:
%===========================================================
%   Semi-Analytical Model of the Bottom and Upper Cells    %
%                of the Atlantic (SAMBUCA)                 %
%                                                          %
%   References: Nikurashin and Vallis, JPO, (2011,2012)    %
%                 Email: man@alum.mit.edu                  %
%===========================================================

clear

%==========================
% Parameters
%==========================
dy = 100e+3;                % meridional grid spacing
dz = 100;                   % vertical grid spacing
dt = 20*24*3600;            % time step

T  = 1000*(12*30*24*3600);  % total integration time

l  =  2e+6;                 % channel width
H  =  4e+3;                 % channel depth
Ly = 15e+6;                 % meridional basin extent
Lx =  6e+6;                 % zonal basin extent

Kv = 2e-5;                  % vertical diffusivity
Kc = 1e+3;                  % convective diffusivity

f  = 1e-4;                  % Coriolis parameter
lambda = 1/(30*24*3600);    % surface restoring rate for buoyancy

Keddy    = 1000;            % eddy diffusivity
bvp_cmin = 0.1;             % BVP eddy parameterization: minimum phase speed
bvp_mode = 2;               % BVP eddy parameterization: baroclinic mode

do_load  = false;           % start from a file
do_save  = true;            % save to a file
fnameIN  = 'out.mat';       % input file name
fnameOUT = 'out.mat';       % output file name
%--------------------------

%==========================
% Grid
%==========================
nt  = T/dt;                 % number of time steps
ny  = l/dy;                 % number of meridional grid points in the channel
nz  = H/dz;                 % number of vertical grid points

nyb = Ly/dy;                % number of meridional grid points in the basin
nyt = ny + nyb;             % total number of meridional grid points

rdy = 1/dy;
rdz = 1/dz;

yb = (-l+dy/2:dy:Ly-dy/2);  % grid for buoyancy
zb = (-H+dz/2:dz:  -dz/2);

yp = (-l+dy:dy:Ly);         % grid for streamfunction
zp = (-H+dz:dz:0 );

[YB,ZB] = meshgrid(yb,zb);
[YP,ZP] = meshgrid(yp,zp);
%--------------------------

%==========================
% Forcing
%==========================
bs = 15*(1+yb(1:ny)'/l) * 2e-4*10;
bn = ( 1.5 - 15/l*(yb(ny+1:end)'-Ly) ) * 2e-4*10;

%wind = (-0.1/1e+3/f)*ones(ny,nz);
%wind = (-0.1/1e+3/f)*(1+yp(1:ny)'/l)*ones(1,nz);
wind = (-0.1/1e+3/f)*sin(pi/2*(1+yp(1:ny)'/l))*ones(1,nz);

kv0 = Kv*ones(1,nz);
%kv0 = 0.5*(1e-3-2e-5)*(1-tanh((zp+2000)*6/800)) + 2e-5;
%kv0 = 1e-3*exp(-(zp+H)/1000)/(1-exp(-H/1000)) + 2e-5;
%--------------------------

%==========================
% Initialization
%==========================
b  = zeros(ny,nz);
p  = zeros(ny,nz);

B  = zeros(nyt,nz);
P  = zeros(nyt,nz);

by = zeros(ny,nz);
bz = zeros(ny,nz);

By = zeros(ny,nz);
Bz = zeros(ny,nz);

v  = zeros(ny,nz);
w  = zeros(ny,nz);

Aud = zeros(ny,nz);
Ans = zeros(ny,nz);
DIV = zeros(ny,nz);
RHS = zeros(ny,nz,3);

TIME = [];
NADW = [];
AABW = [];
%--------------------------

%==========================
% Initialization from file
%==========================
if(do_load)
 disp(['Loading: ',fnameIN]);
 load(fnameIN,'b','RHS');
end
%--------------------------

%==========================
% Implicit diffusion init
%==========================
alpha = dz*dz/dt;

aa = [0, kv0(1:nz-1)];
cc = kv0;
bb = -(alpha+aa+cc);

E0 = diag(bb);
E0 = E0 + diag(aa(2:nz),-1);
E0 = E0 + diag(cc(1:nz-1),1);

E0(1,1)   = -(alpha+kv0(1));
E0(nz,nz) = -(alpha+kv0(nz-1));

E0 = inv(E0);
%--------------------------


for t = 1:nt

%==========================
% Buoyancy gradients
%==========================
by(1:ny-1,:) = (b(2:ny,:)-b(1:ny-1,:))*rdy;
by(ny,:) = 0;
 
bz(:,1:nz-1) = (b(:,2:nz)-b(:,1:nz-1))*rdz;
bz(:,nz) = 0;
%--------------------------

%==========================
% BVP eddy parameterization
%  (Ferrari et al, 2010) 
%==========================
By(:,1:nz-1) = 0.5*(by(:,1:nz-1)+by(:,2:nz));
By(:,nz) = 0;

Bz(1:ny-1,:) = 0.5*(bz(1:ny-1,:)+bz(2:ny,:));
Bz(ny,:) = 0;
 
p = Keddy.*By;
 
%---
c = sum(sqrt(Bz)*dz,2)/(pi*bvp_mode);
c = max(real(c),bvp_cmin);

c = (c/dz).^2;
%---

for n = 1:ny-1
 aa = c(n)*ones(1,nz);
 bb = -(2*c(n)+Bz(n,:));
 cc = c(n)*ones(1,nz);

 E = diag(bb);
 E = E + diag(aa(2:nz),-1);
 E = E + diag(cc(1:nz-1),1);

 r = p(n,:)';

 r(1) = r(1) - c(n)*wind(n,1);

 E(nz,[nz-1 nz])  = [0 1];
 r(nz) = wind(n,nz);

 p(n,:) = E\r;
 
end
 
p = -wind + p;

p(ny,:) = 0;
%--------------------------

%==========================
%         NADW 
%==========================
Bbasin = ones(nyb,1)*b(ny,:);

%---    
BB = bn*ones(1,nz);
Bbasin(Bbasin>BB) = BB(Bbasin>BB);
%---

%---
BB(1:nyb-1,:) = (Bbasin(2:nyb,:)-Bbasin(1:nyb-1,:))*rdy;
BB(nyb,:) = 0;
%---

%---
if(b(ny,nz)~=0) 
    znadw = interp1(b(ny,:),zb,bn(end)); 
else
    znadw=-1; 
end
%---

%---
u = fliplr(cumsum(fliplr(BB)*dz,2))/f;
u = [u(:,2:end) zeros(nyb,1)];

u0 = interp1(zp,'u',znadw);
u = u - u0'*ones(1,nz);
%---

%---
nnadw = find(zp>=znadw);
 
U = 0.5*(u(:,nnadw(1:end-1))+u(:,nnadw(2:end)));
U = sum(U*dz,2);
 
if(znadw~=zp(nnadw(1)))
U = U + 0.5*u(:,nnadw(1))*(zp(nnadw(1))-znadw);
end
 
U = U/(-znadw);
 
u = u - U*ones(1,nz);
%---
 
%--- 
U = fliplr(cumsum(fliplr(0.5*(u(:,1:end-1)+u(:,2:end)))*dz,2));
U = [U zeros(nyb,1)];
%---

%--- 
U = 0.5*(U(1:end-1,:)+U(2:end,:));
U =flipud(cumsum(flipud(U)*dy));
U(end+1,:) = 0;
U(:,zp<znadw) = 0;
%---

U = U/Lx;
 
p(ny,:) = U(1,:);
%--------------------------

%==========================
% Velocities 
%==========================
v(:,1) = -p(:,1)*rdz;
v(:,2:nz) = -(p(:,2:nz)-p(:,1:nz-1))*rdz;
    
w(1,:) =  p(1,:)*rdy;
w(2:ny,:) =  (p(2:ny,:)-p(1:ny-1,:))*rdy;
%--------------------------

%==========================
% Up-stream advection 
%==========================
Ans(1:ny-1,:) = v(1:ny-1,:).*b(2:ny,:);
Ans(v>0) = v(v>0).*b(v>0);
Ans(ny,:) = v(ny,:).*b(ny,:);
 
Aud(:,1:nz-1) = w(:,1:nz-1).*b(:,2:nz);
Aud(w>0) = w(w>0).*b(w>0);
Aud(:,nz) = 0;
%--------------------------

%==========================
% Advective flux divergence 
%==========================
DIV(1,:) = Ans(1,:)*rdy;
DIV(2:ny,:) = (Ans(2:ny,:)-Ans(1:ny-1,:))*rdy;

DIV(:,1) = DIV(:,1) + Aud(:,1)*rdz;
DIV(:,2:nz) = DIV(:,2:nz) + (Aud(:,2:nz)-Aud(:,1:nz-1))*rdz;
%--------------------------

%==========================
% Time stepping (AB2)
%==========================
RHS(:,:,3) = -DIV;

b = b + dt*(3/2*RHS(:,:,3)-1/2*RHS(:,:,2));

RHS(:,:,1) = RHS(:,:,2); 
RHS(:,:,2) = RHS(:,:,3);
%--------------------------

%==========================
% Vertical diffusion
%==========================
sflux = -dz*lambda * (bs-b(:,nz));

for n = 1:ny
    
 E = E0;
 
 nn = find(bz(n,:)<0);

 if (~isempty(nn) || n==ny)

  if(n==ny)
   kv = kv0*Ly/dy;
  else
   kv = kv0;
  end
     
  kv(nn) = Kc;
 
  aa = [0, kv(1:nz-1)];
  cc = kv;
  bb = -(alpha+aa+cc);

  E = diag(bb);
  E = E + diag(aa(2:nz),-1);
  E = E + diag(cc(1:nz-1),1);

  E(1,1)   = -(alpha+kv(1));
  E(nz,nz) = -(alpha+kv(nz-1));

  E = inv(E);
  
 end

 r = -alpha*b(n,:);
 r(nz) = r(nz) + sflux(n)*dz;
 b(n,:) = E*r';
 
end
%--------------------------

%==========================
% Diagnostics 
%==========================
TIME(end+1) = t*dt;
NADW(end+1) = max(p(ny  ,:)) *Lx*1e-6;
AABW(end+1) = min(p(ny-1,:)) *Lx*1e-6;
%--------------------------

fprintf('Completed: %g%% NADW: %4.2f AABW: %4.2f \n',round(100*t/nt),NADW(end),AABW(end));

end

%==========================
% Matching & Interpolation 
%==========================
B = [b; Bbasin];

nn = find(diff(U(:,nz-1))==0);
 
if(~isempty(nn))
  nn = nn(end) + 1;
else
  nn = ny + 1;
end

P(1:ny-1,:) =  p(1:ny-1,:);
P(ny+nn:end,nnadw) = U(nn:end,nnadw);

slp = ones(nn,1) * (p(ny,nnadw)-p(ny-1,nnadw)) / yp(ny+nn-1);
sft = ones(nn,1) * p(ny-1,nnadw);
P(ny:ny+nn-1,nnadw) = slp .* YP(nnadw,ny:ny+nn-1)' + sft;

if(~isempty(nnadw))
naabw = (1:nnadw(1)-1);
else
naabw = (1:nz);
end

slp = ones(nyb,1) * (p(ny,naabw)-p(ny-1,naabw)) / Ly;
sft = ones(nyb,1) * p(ny-1,naabw);
P(ny:nyt-1,naabw) = slp .* YP(naabw,ny:nyt-1)' + sft;   
%--------------------------

%==========================
% Plot
%==========================
figure

%load blue_red.mat
%colormap(map);

cc = [-15:0.5:-0.5,-0.1, 1:15];
[c,h] = contourf(yp*1e-3,zp,'P'*Lx*1e-6,cc,'linestyle','--');
shading flat;
caxis([cc(1) cc(end)]);
colorbar

hold on

cc = [1:10];
[c,h] = contour(yb*1e-3,zb,'B'/2e-4/10,cc,'color','k','linewidth',0.5);
clabel(c,h,cc,'fontsize',8,'color','k');

xlim([-l Ly]*1e-3);
ylim([-H 0]);

set(gca,'fontsize',12)
xlabel('Distance, (km)')
ylabel('Depth, (m)');
title(sprintf('Temperature in (C) and streamfunction in (Sv). Time: %g yr',round(t*dt/12/30/24/3600)));

hold off
%--------------------------

if(do_save); disp(['Saving: ',fnameOUT]); save(fnameOUT); end


SyntaxError: unterminated string literal (detected at line 67) (1225329659.py, line 67)